### Creating CollectionBuilder compatible metadata improvements

- changing object_id to objectid
- adding 'filename' column as /article_id/file (removing objects/ in front)

In [ ]:
import os
import pandas as pd
import re


In [ ]:

# --- CONFIG ---
''' 

INFILE = "complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "cb_complete_metadata_images_tropes_reprints_transcripts.csv"  # safer new file
OVERWRITE = False  # set True to write back into INFILE
# ---------------

df = pd.read_csv(INFILE, dtype=str).fillna("")

# 1) Ensure CollectionBuilder-friendly objectid
if "objectid" not in df.columns:
    if "object_id" in df.columns:
        df = df.rename(columns={"object_id": "objectid"})
        print("Renamed column: object_id -> objectid")
    else:
        raise ValueError("Could not find 'objectid' or 'object_id' column in CSV.")

# 2) Add filename (relative to /objects/)
# Prefer image_object_location (your example); fall back to object_location if you use that.
path_col = None
for candidate in ["filename", "image_object_location", "object_location", "file", "filepath"]:
    if candidate in df.columns:
        path_col = candidate
        break

if "filename" not in df.columns:
    if path_col in ["image_object_location", "object_location"]:
        def to_filename(p: str) -> str:
            p = (p or "").strip()
            if not p:
                return ""
            # normalize slashes
            p = p.replace("\\", "/")
            # if it starts with objects/, strip it
            if p.startswith("objects/"):
                return p[len("objects/"):]
            # if it starts with /objects/, strip it
            if p.startswith("/objects/"):
                return p[len("/objects/"):]
            # otherwise assume it's already relative to objects/
            return p

        df["filename"] = df[path_col].apply(to_filename)
        print(f"Created 'filename' from '{path_col}' by stripping leading 'objects/'")
    elif path_col == "filename":
        # already exists, but this branch won't happen because we checked above
        pass
    else:
        raise ValueError(
            "Could not find a usable path column. Expected one of: "
            "'image_object_location' or 'object_location' (or already have 'filename')."
        )
else:
    print("Column 'filename' already exists—leaving as-is.")

# 3) Sanity checks: show a few examples
print("\nSample (objectid, filename):")
print(df[["objectid", "filename"]].head(10).to_string(index=False))

# 4) Write output
outfile = INFILE if OVERWRITE else OUTFILE
df.to_csv(outfile, index=False)
print(f"\nWrote: {outfile}")

'''


In [ ]:
'''

orig = pd.read_csv("complete_metadata_images_tropes_reprints_transcripts.csv", dtype=str).fillna("")
new  = pd.read_csv("_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv", dtype=str).fillna("")

print("Original rows:", len(orig))
print("New rows     :", len(new))

# normalize column name for checking
if "objectid" not in orig.columns and "object_id" in orig.columns:
    orig = orig.rename(columns={"object_id": "objectid"})
if "objectid" not in new.columns and "object_id" in new.columns:
    new = new.rename(columns={"object_id": "objectid"})

print("\nOriginal duplicate objectid count:",
      orig["objectid"].duplicated().sum())
print("New duplicate objectid count:",
      new["objectid"].duplicated().sum())

# Show some duplicated IDs from the new file, if any:
dupes = new[new["objectid"].duplicated(keep=False)].sort_values("objectid")
print("\nExample duplicated rows in NEW (first 30):")
cols = [c for c in ["objectid","article_id","image_display_template","image_object_location","filename"] if c in new.columns]
print(dupes[cols].head(30).to_string(index=False))

'''


### Solving issue of duplicate rows

In [6]:


INFILE  = "complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"

# --- helper functions ---
def slug(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", "_", s)
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return s

def to_filename(p: str) -> str:
    p = (p or "").strip().replace("\\", "/")
    if not p:
        return ""
    if p.startswith("objects/"):
        return p[len("objects/"):]
    if p.startswith("/objects/"):
        return p[len("/objects/"):]
    return p

def img_index_from_filename(fn: str) -> str:
    """
    Try to extract a nice image index:
    - Warner1856_LAS_DigSur_12.png -> 12
    - fallback: use the whole filename slug if needed
    """
    fn = (fn or "").strip()
    m = re.search(r"(\d+)(?=\.[A-Za-z0-9]+$)", fn)
    return m.group(1) if m else ""

# --- load ---
df = pd.read_csv(INFILE, dtype=str).fillna("")
n0 = len(df)

# --- required columns ---
required = ["article_id", "image_display_template"]
for c in required:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# group context column(s)
# Your key varying ID is group_reprint_id (per your message).
if "group_reprint_id" not in df.columns:
    raise ValueError("Expected column 'group_reprint_id' (you said you have it).")

# Create filename if needed
if "filename" not in df.columns:
    if "image_object_location" not in df.columns:
        raise ValueError("Need 'image_object_location' to derive filename.")
    df["filename"] = df["image_object_location"].apply(to_filename)

# Normalize a "context" token:
# - Use group_reprint_id when present (this distinguishes duplicated rows)
# - Else fallback to article_id so non-duplicated rows remain stable
df["_article"]  = df["article_id"].apply(slug)
df["_context"]  = df["group_reprint_id"].apply(slug)
df.loc[df["_context"].str.strip() == "", "_context"] = df.loc[df["_context"].str.strip() == "", "_article"]

# OPTIONAL: also incorporate reprint_type into context for extra uniqueness
USE_REPRINT_TYPE_IN_CONTEXT = True
if USE_REPRINT_TYPE_IN_CONTEXT and "reprint_type" in df.columns:
    rt = df["reprint_type"].apply(slug)
    df["_context"] = df["_context"] + "__" + rt.where(rt.str.strip() != "", "na")

# Identify parent vs image rows
tmpl = df["image_display_template"].str.lower().str.strip()
is_parent = tmpl.eq("compound_object")
is_image  = tmpl.eq("image")

# Build parent objectid
df.loc[is_parent, "objectid"] = df.loc[is_parent].apply(
    lambda r: f"{r['_article']}__{r['_context']}",
    axis=1
)

# Build lookup for parent objectids by (article, context)
parent_lookup = {
    (r["_article"], r["_context"]): r["objectid"]
    for _, r in df[is_parent].iterrows()
}

# Build image objectids + set image_parent_id to correct parent
def build_img_objectid(row):
    idx = ""
    # If you have an explicit image number column, prefer it
    for col in ["image_num", "img_num", "sequence", "order", "page"]:
        if col in df.columns and str(row[col]).strip():
            idx = str(row[col]).strip()
            break
    if not idx:
        idx = img_index_from_filename(row["filename"])
    if not idx:
        # last resort: unique-ish from filename itself
        idx = slug(row["filename"])
    return f"{row['_article']}__{row['_context']}__img{idx}"

def parent_for_image(row):
    key = (row["_article"], row["_context"])
    return parent_lookup.get(key, "")

df.loc[is_image, "objectid"] = df.loc[is_image].apply(build_img_objectid, axis=1)
df.loc[is_image, "image_parent_id"] = df.loc[is_image].apply(parent_for_image, axis=1)

# Clean temp columns
df = df.drop(columns=["_article", "_context"])

# Sanity checks
assert len(df) == n0, "Row count changed unexpectedly—abort!"

dupes = df[df["objectid"].duplicated(keep=False)]
print("Rows:", len(df))
print("Duplicate objectid rows after rebuild:", len(dupes))

# Helpful diagnostics if any dupes remain
if len(dupes) > 0:
    cols = [c for c in ["objectid","article_id","group_reprint_id","reprint_type","image_display_template","filename","image_parent_id"] if c in df.columns]
    print(dupes[cols].head(40).to_string(index=False))

# Write out
df.to_csv(OUTFILE, index=False)
print("Wrote:", OUTFILE)


Rows: 4428
Duplicate objectid rows after rebuild: 0
Wrote: _data/cb_complete_metadata_images_tropes_reprints_transcripts.csv


### Adding object_location column for CB 

In [7]:


INFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"  # <-- change if needed
OUTFILE = INFILE  # overwrite

df = pd.read_csv(INFILE, dtype=str).fillna("")

def make_object_location(row):
    fn = (row.get("filename") or "").strip()
    if fn:
        return "/objects/" + fn
    loc = (row.get("image_object_location") or "").strip()
    if loc:
        # normalize to leading slash
        loc = loc.replace("\\", "/")
        return loc if loc.startswith("/") else "/" + loc
    return ""

df["object_location"] = df.apply(make_object_location, axis=1)

# (Optional but often helpful) Set format so CB routes layout correctly
if "format" not in df.columns:
    df["format"] = ""
mask_img = df.get("image_display_template", "").str.lower().eq("image")
df.loc[mask_img, "format"] = "image"

df.to_csv(OUTFILE, index=False)
print("Added object_location (+ format=image for image rows).")


Added object_location (+ format=image for image rows).


### Adding diplay_template column

In [8]:


INFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")

df["display_template"] = df["image_display_template"]

df.to_csv(OUTFILE, index=False)
print("Added display_template column.")


Added display_template column.


## Adding image format column

In [10]:
import pandas as pd

INFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")
if "format" not in df.columns:
    df["format"] = ""

tmpl = df["image_display_template"].str.lower().str.strip()
df.loc[tmpl.eq("image"), "format"] = "image"
df.loc[tmpl.eq("compound_object"), "format"] = "multiple"  # CB often uses multiple.html for compound

df.to_csv(OUTFILE, index=False)
print("Set format for image + compound rows.")


Set format for image + compound rows.


In [ ]:
### changing object_id column to object_image_tag



In [ ]:

INFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"
OUTFILE = INFILE

df = pd.read_csv(INFILE, dtype=str).fillna("")

if "object_id" in df.columns:
    df = df.rename(columns={"object_id": "article_image_tag"})
    print("Renamed column: object_id -> article_image_tag")
else:
    print("Column 'object_id' not found — no rename performed.")

print("Columns:", df.columns.tolist())
df.to_csv(OUTFILE, index=False)
print("Wrote:", OUTFILE)


### Articles appearing in multiple document groups (many-to-many: article_id ↔ group_reprint_id)

In [6]:

INFILE = "_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv"

df = pd.read_csv(INFILE, dtype=str).fillna("")

# Use only compound_object rows — one row per (article, document group) pair
# This avoids counting image child rows as separate occurrences
compound = df[df["image_display_template"].str.lower().str.strip() == "compound_object"].copy()

# For each article_id, collect the distinct document groups it appears in
article_groups = (
    compound.groupby("article_id")["group_reprint_id"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .reset_index()
    .rename(columns={"group_reprint_id": "document_groups"})
)
article_groups["num_groups"] = article_groups["document_groups"].apply(len)

# Filter to articles that appear in MORE than one document group
multi = article_groups[article_groups["num_groups"] > 1].sort_values("num_groups", ascending=False)

print(f"Total articles: {len(article_groups)}")
print(f"Articles in multiple document groups: {len(multi)}\n")

# Show the full breakdown
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
print(multi[["article_id", "num_groups", "document_groups"]].to_string(index=False))


Total articles: 481
Articles in multiple document groups: 13

            article_id  num_groups                                      document_groups
    CambriaFreeman1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
 SalemWeeklyReview1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
          Sheridan1925           2            [Sheridan1925_reprint, Terry1882_reprint]
   Sheridan1927_21(33)           2            [Sheridan1925_reprint, Terry1882_reprint]
   Sheridan1927_21(34)           2            [Sheridan1925_reprint, Terry1882_reprint]
ShipwreckedMariner1879           2  [SalemWeeklyReview1879_reprint, Stuart1878_reprint]
     Taylor1860_12(11)           2 [Russell1856_HCF_reprint, Taylor1860_12(11)_reprint]
     Taylor1860_13(16)           2 [Russell1856_HCF_reprint, Taylor1860_12(11)_reprint]
        TheGazette1879           2         [Stuart1878_reprint, TheGazette1879_reprint]
 TheNewfoundlander1879           2         [Stuart1878_rep